In [ ]:
"""
Embedding Clustering Analysis with GMM

This notebook clusters embeddings using Gaussian Mixture Models (GMM) 
with 6 clusters corresponding to the 6 VAD poles:
- pos_V, neg_V (Valence)
- pos_A, neg_A (Arousal)  
- pos_D, neg_D (Dominance)

Evaluation metrics:
- Purity: How pure each cluster is with respect to ground truth
- Silhouette Score: How well-separated the clusters are
- NMI (Normalized Mutual Information): Mutual information between predicted and true labels
"""

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Base directories
datasets_dir = Path('../datasets/embeddings')
results_dir = Path('../results')

# Languages to process
languages = ['arabic', 'english']
lang_codes = {'arabic': 'ar', 'english': 'en'}

# Model configurations (model_name: filename_prefix)
models = {
    'BERT': 'bert',
    'RoBERTa': 'roberta',
    'NeoBERT': 'neobert',
    'ModernBERT': 'modernbert',
    'AraBERT-base': 'arabert_base',
    'AraBERT-large': 'arabert_large',
    'BGE-M3': 'bge_m3',
    'E5-large-v2': 'e5_large_v2',
    'Gemma-300M': 'gemma_300m',
    'OpenAI-3-large': 'openai_3_large',
    'OpenAI-ada': 'openai_ada',
    'Voyage-3-large': 'voyage_3_large'
}

# GMM configuration
N_CLUSTERS = 6  # 6 poles: {pos, neg} × {V, A, D}
N_INIT = 10     # Number of initializations
RANDOM_STATE = 42

print(f"Models to analyze: {len(models)}")
print(f"Languages: {languages}")
print(f"Number of clusters: {N_CLUSTERS}")
print(f"Total configurations: {len(models) * len(languages)}")


In [ ]:
def load_embeddings(filepath):
    """Load embeddings from CSV file and extract embedding columns."""
    df = pd.read_csv(filepath)
    emb_cols = [col for col in df.columns if col.startswith('emb_')]
    return df, emb_cols


def get_embedding_filepath(language, model_prefix):
    """Construct the filepath for a given language and model."""
    lang_code = lang_codes[language]
    filename = f"{model_prefix}_{lang_code}.csv"
    return datasets_dir / language / 'poles' / filename


def create_ground_truth_labels(df):
    """Create ground truth pole labels from pos/neg and V/A/D columns."""
    labels = df['pos/neg'] + '_' + df['V/A/D']
    return labels.values


In [ ]:
def calculate_purity(y_true, y_pred):
    """
    Calculate clustering purity.
    
    Purity measures how pure each cluster is - i.e., the proportion of 
    the most common ground truth class in each cluster, weighted by cluster size.
    
    Purity = (1/N) * Σ max_j |cluster_i ∩ class_j|
    
    Returns a value between 0 and 1, where 1 = perfect purity.
    """
    # Encode labels if they are strings
    if isinstance(y_true[0], str):
        le = LabelEncoder()
        y_true_encoded = le.fit_transform(y_true)
    else:
        y_true_encoded = y_true
    
    # Build contingency matrix
    contingency = np.zeros((len(np.unique(y_pred)), len(np.unique(y_true_encoded))))
    
    for i, (pred, true) in enumerate(zip(y_pred, y_true_encoded)):
        contingency[pred, true] += 1
    
    # Sum of max in each row (cluster)
    purity = np.sum(np.max(contingency, axis=1)) / len(y_true)
    
    return purity


In [ ]:
def evaluate_gmm_clustering(embeddings, ground_truth_labels, n_clusters=6, n_init=10, random_state=42):
    """
    Perform GMM clustering and evaluate against ground truth.
    
    Args:
        embeddings: numpy array of shape (n_samples, n_features)
        ground_truth_labels: array of ground truth class labels
        n_clusters: number of GMM components (default 6 for VAD poles)
        n_init: number of initializations
        random_state: random seed for reproducibility
    
    Returns:
        dict with clustering metrics and predictions
    """
    # Fit GMM
    gmm = GaussianMixture(
        n_components=n_clusters,
        covariance_type='full',
        n_init=n_init,
        random_state=random_state
    )
    
    cluster_labels = gmm.fit_predict(embeddings)
    
    # Calculate metrics
    purity = calculate_purity(ground_truth_labels, cluster_labels)
    
    # Silhouette score (internal clustering quality)
    silhouette = silhouette_score(embeddings, cluster_labels)
    
    # NMI (Normalized Mutual Information)
    le = LabelEncoder()
    true_labels_encoded = le.fit_transform(ground_truth_labels)
    nmi = normalized_mutual_info_score(true_labels_encoded, cluster_labels)
    
    return {
        'purity': purity,
        'silhouette': silhouette,
        'nmi': nmi,
        'cluster_labels': cluster_labels,
        'gmm_model': gmm,
        'bic': gmm.bic(embeddings),
        'aic': gmm.aic(embeddings)
    }


In [ ]:
# Store results per language
results_by_language = {lang: {} for lang in languages}

# Process all models for both languages
for language in languages:
    print(f"\n{'='*60}")
    print(f"Processing {language.upper()} embeddings")
    print(f"{'='*60}")
    
    for model_name, model_prefix in models.items():
        filepath = get_embedding_filepath(language, model_prefix)
        
        if not filepath.exists():
            print(f"  ⚠️  {model_name}: File not found - {filepath}")
            continue
            
        print(f"  Processing {model_name}...", end=" ")
        
        try:
            # Load data
            df, emb_cols = load_embeddings(filepath)
            embeddings = df[emb_cols].values
            ground_truth = create_ground_truth_labels(df)
            
            # Evaluate GMM clustering
            eval_results = evaluate_gmm_clustering(
                embeddings, 
                ground_truth,
                n_clusters=N_CLUSTERS,
                n_init=N_INIT,
                random_state=RANDOM_STATE
            )
            
            results_by_language[language][model_name] = {
                'filepath': str(filepath),
                'embedding_dim': len(emb_cols),
                'num_samples': len(df),
                'purity': eval_results['purity'],
                'silhouette': eval_results['silhouette'],
                'nmi': eval_results['nmi'],
                'bic': eval_results['bic'],
                'aic': eval_results['aic'],
                'cluster_labels': eval_results['cluster_labels'],
                'ground_truth': ground_truth
            }
            
            print(f"✓ Purity: {eval_results['purity']:.4f}, Silhouette: {eval_results['silhouette']:.4f}, NMI: {eval_results['nmi']:.4f}")
            
        except Exception as e:
            print(f"✗ Error: {e}")

print(f"\n{'='*60}")
print("Processing complete!")
print(f"{'='*60}")


In [ ]:
# Create clustering metrics DataFrames per language
clustering_dfs = {}

for language in languages:
    print(f"\n{'='*60}")
    print(f"{language.upper()} - Clustering Metrics (GMM with {N_CLUSTERS} clusters)")
    print(f"{'='*60}")
    print("Higher values are better for all metrics")
    print()
    
    if not results_by_language[language]:
        print("No results available for this language.")
        continue
    
    # Create DataFrame with metrics
    metrics_data = {
        model: {
            'purity': results['purity'],
            'silhouette': results['silhouette'],
            'nmi': results['nmi'],
            'embedding_dim': results['embedding_dim']
        }
        for model, results in results_by_language[language].items()
    }
    
    clustering_df = pd.DataFrame(metrics_data).T
    
    # Add combined score (average of normalized metrics)
    clustering_df['combined_score'] = (clustering_df['purity'] + clustering_df['silhouette'] + clustering_df['nmi']) / 3
    
    # Sort by combined score
    clustering_df = clustering_df.sort_values('combined_score', ascending=False)
    clustering_dfs[language] = clustering_df
    
    # Display
    display_cols = ['purity', 'silhouette', 'nmi', 'combined_score', 'embedding_dim']
    print(clustering_df[display_cols].round(4).to_string())
    print()
    
    # Best model
    best_model = clustering_df['combined_score'].idxmax()
    print(f"🏆 Best clustering performance: {best_model}")
    print(f"   Purity: {clustering_df.loc[best_model, 'purity']:.4f}")
    print(f"   Silhouette: {clustering_df.loc[best_model, 'silhouette']:.4f}")
    print(f"   NMI: {clustering_df.loc[best_model, 'nmi']:.4f}")


In [ ]:
# Save results per language
for language in languages:
    print(f"\n{'='*60}")
    print(f"Saving {language.upper()} results")
    print(f"{'='*60}")
    
    if not results_by_language[language]:
        print("No results to save for this language.")
        continue
    
    lang_results_dir = results_dir / language
    lang_results_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Save clustering metrics
    metrics_path = lang_results_dir / 'clustering_metrics.csv'
    clustering_dfs[language].to_csv(metrics_path)
    print(f"  ✓ Clustering metrics saved: {metrics_path}")
    
    # 2. Save cluster assignments for each model
    cluster_rows = []
    for model_name, results in results_by_language[language].items():
        cluster_labels = results['cluster_labels']
        ground_truth = results['ground_truth']
        
        for i, (cluster, truth) in enumerate(zip(cluster_labels, ground_truth)):
            cluster_rows.append({
                'model': model_name,
                'sample_idx': i,
                'cluster_label': cluster,
                'ground_truth': truth
            })
    
    clusters_df = pd.DataFrame(cluster_rows)
    clusters_path = lang_results_dir / 'cluster_assignments.csv'
    clusters_df.to_csv(clusters_path, index=False)
    print(f"  ✓ Cluster assignments saved: {clusters_path}")
    print(f"    Rows: {len(clusters_df)} ({len(results_by_language[language])} models × samples)")

print(f"\n{'='*60}")
print("All results saved successfully!")
print(f"{'='*60}")


In [ ]:
# Summary and comparison between languages
print("="*70)
print("FINAL SUMMARY: GMM Clustering Analysis Results")
print("="*70)

print(f"\nModels analyzed: {len(models)}")
print(f"Languages: {', '.join(languages)}")
print(f"Clusters: {N_CLUSTERS} (6 VAD poles)")
print(f"Total configurations: {sum(len(r) for r in results_by_language.values())}")

# Per-language rankings
for language in languages:
    if language not in clustering_dfs or clustering_dfs[language].empty:
        continue
        
    print(f"\n{'─'*70}")
    print(f"📊 {language.upper()} Rankings (by combined score, higher = better)")
    print(f"{'─'*70}")
    
    ranking = clustering_dfs[language].sort_values('combined_score', ascending=False)
    for i, (model, row) in enumerate(ranking.iterrows(), 1):
        emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "  "
        print(f"{emoji} {i:2d}. {model:<20} | Purity: {row['purity']:.4f} | Silhouette: {row['silhouette']:.4f} | NMI: {row['nmi']:.4f}")

# Cross-language comparison
print(f"\n{'─'*70}")
print("🌐 Cross-Language Best Models Comparison")
print(f"{'─'*70}")

for language in languages:
    if language not in clustering_dfs or clustering_dfs[language].empty:
        continue
    best = clustering_dfs[language]['combined_score'].idxmax()
    row = clustering_dfs[language].loc[best]
    print(f"  {language.capitalize()}: {best}")
    print(f"    Purity: {row['purity']:.4f}, Silhouette: {row['silhouette']:.4f}, NMI: {row['nmi']:.4f}")

print(f"\n{'='*70}")
print("Analysis complete. Results saved to ../results/{language}/ directories.")
print("="*70)


In [ ]:
# Display saved files structure
print("Output files structure:")
for language in languages:
    lang_results_dir = results_dir / language
    if lang_results_dir.exists():
        print(f"\n{language}/")
        for f in sorted(lang_results_dir.glob("*.csv")):
            size_kb = f.stat().st_size / 1024
            print(f"  └── {f.name} ({size_kb:.1f} KB)")
